## Matrix Multiplication

In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpnfjx4jma".


In [14]:
%%cuda
#include <stdio.h>
#include <iostream>

__global__ void matrixMul(int *A, int *B, int *C, int N)
{
    int row = blockIdx.y *blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < N and col < N)
    {
        int sum = 0;
        for(int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k*N+col];
        }
        __syncthreads();
        C[row * N + col] = sum; 
    }
}

void cpuMatrixMul(int *A, int *B, int *C, int N)
{
    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            int sum = 0;
            for(int k = 0; k < N; k++)
            {
                sum += A[i * N + k] * B[k*N+j];
            }
            C[i * N + j] = sum;
        }
    }
}

int main()
{
    int N = 1024;

    int *h_A, *h_B, *h_C, *h_gpu_c;
    h_A = (int*)malloc(N*N*sizeof(int));
    h_B = (int*)malloc(N*N*sizeof(int));
    h_C = (int*)malloc(N*N*sizeof(int));
    h_gpu_c = (int*)malloc(N*N*sizeof(int));

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N * N * sizeof(int));
    cudaMalloc(&d_B, N * N * sizeof(int));
    cudaMalloc(&d_C, N * N * sizeof(int));

std::cout<<"Memory allocated on device"<<std::endl;
        for(int i = 0; i < N; i++)
        {
            for(int j = 0; j < N; j++)
            {
                h_A[i*N + j] = rand() % 10;
                h_B[i*N + j] = rand() % 10;
                h_C[i*N + j] = 0;
            }
        }
        clock_t start = clock();
        cpuMatrixMul(h_A, h_B, h_C, N);
        clock_t end = clock();
        double cpu_time = double(end - start) * 1000 / CLOCKS_PER_SEC;
        std::cout<<"CPU Time: "<<cpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(d_A, h_A, N*N*sizeof(int), cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_B, N*N*sizeof(int), cudaMemcpyHostToDevice);

        dim3 blocksize(16, 16);
        dim3 gridsize((N + blocksize.x - 1)/blocksize.x, (N + blocksize.y - 1)/blocksize.y);
        cudaEvent_t start_event, stop_event;
        cudaEventCreate(&start_event);
        cudaEventCreate(&stop_event);
        cudaEventRecord(start_event, 0);
        matrixMul<<<gridsize, blocksize>>>(d_A, d_B, d_C, N);
        cudaDeviceSynchronize();
        cudaEventRecord(stop_event, 0);
        cudaEventSynchronize(stop_event);
        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start_event, stop_event);
        std::cout<<"GPU Time: "<<gpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(h_gpu_c, d_C, N*N*sizeof(int), cudaMemcpyDeviceToHost);

        //compare results
        long temp = 0;
        for(int i = 0; i < N; i++)
        {
            for(int j = 0; j < N; j++)
            {
                temp += abs(h_gpu_c[i*N + j] - h_C[i*N + j]);
            }
        }

        if(temp == 0)
        {
            std::cout<<"Results are correct"<<std::endl;
        }
        else
        {
            std::cout<<"Results are incorrect"<<std::endl;
        }

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        free(h_A);
        free(h_B);
        free(h_C);
        free(h_gpu_c);

        return 0;

}

Memory allocated on device
CPU Time: 8099.72 milliseconds
GPU Time: 9.48771 milliseconds
Results are correct



## Matrix multiplication using Shared memory

In [2]:
%%cuda
#include <stdio.h>
#include <iostream>

#define TILE_SIZE 16


__global__ void matrixMul(int *A, int *B, int *C, int N)
{
    __shared__ int shared_a[TILE_SIZE][TILE_SIZE];
    __shared__ int shared_b[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y *TILE_SIZE + threadIdx.y;
    int col = blockIdx.x *TILE_SIZE + threadIdx.x;
    for(int t = 0; t < N/TILE_SIZE; t++)
    {
        shared_a[threadIdx.y][threadIdx.x] = A[row * N + (t*TILE_SIZE + threadIdx.x)];
        shared_b[threadIdx.y][threadIdx.x] = B[(t*TILE_SIZE + threadIdx.y)*N + col];
        __syncthreads();

        for(int k = 0; k < TILE_SIZE; k++)
        {
            C[row * N + col] += shared_a[threadIdx.y][k] * shared_b[k][threadIdx.x];
            __syncthreads();
        }
    }
}

void cpuMatrixMul(int *A, int *B, int *C, int N)
{
    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            int sum = 0;
            for(int k = 0; k < N; k++)
            {
                sum += A[i * N + k] * B[k*N+j];
            }
            C[i * N + j] = sum;
        }
    }
}

int main()
{
    int N = 1024;

    int *h_A, *h_B, *h_C, *h_gpu_c;
    h_A = (int*)malloc(N*N*sizeof(int));
    h_B = (int*)malloc(N*N*sizeof(int));
    h_C = (int*)malloc(N*N*sizeof(int));
    h_gpu_c = (int*)malloc(N*N*sizeof(int));

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N * N * sizeof(int));
    cudaMalloc(&d_B, N * N * sizeof(int));
    cudaMalloc(&d_C, N * N * sizeof(int));

std::cout<<"Memory allocated on device"<<std::endl;
        for(int i = 0; i < N; i++)
        {
            for(int j = 0; j < N; j++)
            {
                h_A[i*N + j] = rand() % 10;
                h_B[i*N + j] = rand() % 10;
                h_C[i*N + j] = 0;
            }
        }
        clock_t start = clock();
        cpuMatrixMul(h_A, h_B, h_C, N);
        clock_t end = clock();
        double cpu_time = double(end - start) * 1000 / CLOCKS_PER_SEC;
        std::cout<<"CPU Time: "<<cpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(d_A, h_A, N*N*sizeof(int), cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_B, N*N*sizeof(int), cudaMemcpyHostToDevice);

        dim3 blocksize(16, 16);
        dim3 gridsize((N + blocksize.x - 1)/blocksize.x, (N + blocksize.y - 1)/blocksize.y);
        cudaEvent_t start_event, stop_event;
        cudaEventCreate(&start_event);
        cudaEventCreate(&stop_event);
        cudaEventRecord(start_event, 0);
        matrixMul<<<gridsize, blocksize>>>(d_A, d_B, d_C, N);
        cudaDeviceSynchronize();
        cudaEventRecord(stop_event, 0);
        cudaEventSynchronize(stop_event);
        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start_event, stop_event);
        std::cout<<"GPU Time: "<<gpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(h_gpu_c, d_C, N*N*sizeof(int), cudaMemcpyDeviceToHost);

        //compare results
        long temp = 0;
        for(int i = 0; i < N; i++)
        {
            for(int j = 0; j < N; j++)
            {
                temp += abs(h_gpu_c[i*N + j] - h_C[i*N + j]);
            }
        }

        if(temp == 0)
        {
            std::cout<<"Results are correct"<<std::endl;
        }
        else
        {
            std::cout<<"Results are incorrect"<<std::endl;
        }

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        free(h_A);
        free(h_B);
        free(h_C);
        free(h_gpu_c);

        return 0;

}

Memory allocated on device
CPU Time: 7694.99 milliseconds
GPU Time: 127.847 milliseconds
Results are correct



## Matrix Multiplication using float4

In [13]:
%%cuda
#include <stdio.h>
#include <iostream>

#define TILE_WIDTH 16


__global__ void mmTiledVec4(const float* __restrict__ A,
                            const float* __restrict__ B,
                            float*       __restrict__ C,
                            int n)
{
    // Shared memory for one tile (16×16) of A and one tile (16×16) of B
    __shared__ float sA[TILE_WIDTH][TILE_WIDTH];
    __shared__ float sB[TILE_WIDTH][TILE_WIDTH];

    // Identify which row & column(s) this thread will handle
    // blockIdx.x, blockIdx.y => which tile
    // threadIdx.x in [0..3], threadIdx.y in [0..15]
    int Row = blockIdx.y * TILE_WIDTH + threadIdx.y;
    // Each thread covers 4 columns:
    //   baseCol = blockIdx.x*16 + threadIdx.x*4
    int baseCol = blockIdx.x * TILE_WIDTH + (threadIdx.x * 4);

    // We'll accumulate partial sums for these 4 columns in a small array
    float sum[4] = {0.0f, 0.0f, 0.0f, 0.0f};

    // how many 16×16 tiles we must iterate over in the K dimension
    int numTiles = n / TILE_WIDTH;  // assume n is multiple of 16

    // For each tile in [0..numTiles-1]
    for (int t = 0; t < numTiles; t++)
    {
        // --------------------------------------------------
        // 1. Load sub-tile of A and B into shared memory
        //    Each thread loads a float4 from A, and a float4 from B
        // --------------------------------------------------

        // A
        //sA[threadIdx.y][threadIdx.x] = A[row * n + (t * TILE_SIZE + threadIdx.x)];
        {   //
            // global col for A => t*TILE_WIDTH + (threadIdx.x*4 .. +3)
            int globalACol = t*TILE_WIDTH + (threadIdx.x * 4);
            const float* srcA = A + (Row*n + globalACol);
            float* dstA = &sA[threadIdx.y][threadIdx.x*4];

            float4 vecA = *reinterpret_cast<const float4*>(srcA);
            *reinterpret_cast<float4*>(dstA) = vecA;
        }

        //sB[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * N + col];
        // B
        {
            // global row for B => t*TILE_WIDTH + threadIdx.y
            int globalBRow = t*TILE_WIDTH + threadIdx.y;
            // columns => baseCol..baseCol+3
            const float* srcB = B + (globalBRow*n + baseCol);
            float* dstB = &sB[threadIdx.y][threadIdx.x*4];

            float4 vecB = *reinterpret_cast<const float4*>(srcB);
            *reinterpret_cast<float4*>(dstB) = vecB;
        }

        __syncthreads();

        // --------------------------------------------------
        // 2. Compute partial sums
        // --------------------------------------------------
        for(int k = 0; k < TILE_WIDTH; k++){
            float aVal = sA[threadIdx.y][k];
            float4 bVal4 = *reinterpret_cast<float4*>(&sB[k][threadIdx.x*4]);
            sum[0] += aVal * bVal4.x;
            sum[1] += aVal * bVal4.y;
            sum[2] += aVal * bVal4.z;
            sum[3] += aVal * bVal4.w;
        }

        __syncthreads();
    }

    // --------------------------------------------------
    // 3. Write out final partial sums
    // --------------------------------------------------
    if(Row < n){
        // We skip boundary checks for columns. If baseCol+3 < n => safe
        float* outC = C + (Row*n + baseCol);
        outC[0] = sum[0];
        outC[1] = sum[1];
        outC[2] = sum[2];
        outC[3] = sum[3];
    }
}

void cpuMatrixMul(float *A, float *B, float *C, int N)
{
    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            float sum = 0;
            for(int k = 0; k < N; k++)
            {
                sum += A[i * N + k] * B[k*N+j];
            }
            C[i * N + j] = sum;
        }
    }
}

int main()
{
    int N = 1024;

    float *h_A, *h_B, *h_C, *h_gpu_c;
    h_A = (float*)malloc(N*N*sizeof(float));
    h_B = (float*)malloc(N*N*sizeof(float));
    h_C = (float*)malloc(N*N*sizeof(float));
    h_gpu_c = (float*)malloc(N*N*sizeof(float));

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N * N * sizeof(float));
    cudaMalloc(&d_B, N * N * sizeof(float));
    cudaMalloc(&d_C, N * N * sizeof(float));

std::cout<<"Memory allocated on device"<<std::endl;
        for(int i = 0; i < N; i++)
        {
            for(int j = 0; j < N; j++)
            {
                h_A[i*N + j] = rand() % 10;
                h_B[i*N + j] = rand() % 10;
                h_C[i*N + j] = 0;
            }
        }
        clock_t start = clock();
        cpuMatrixMul(h_A, h_B, h_C, N);
        clock_t end = clock();
        double cpu_time = double(end - start) * 1000 / CLOCKS_PER_SEC;
        std::cout<<"CPU Time: "<<cpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(d_A, h_A, N*N*sizeof(float), cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, h_B, N*N*sizeof(float), cudaMemcpyHostToDevice);

        dim3 blocksize(4, 16);
        dim3 gridsize((N/TILE_WIDTH, N/TILE_WIDTH));
        cudaEvent_t start_event, stop_event;
        cudaEventCreate(&start_event);
        cudaEventCreate(&stop_event);
        cudaEventRecord(start_event, 0);
        mmTiledVec4<<<gridsize, blocksize>>>(d_A, d_B, d_C, N);
        cudaDeviceSynchronize();
        cudaEventRecord(stop_event, 0);
        cudaEventSynchronize(stop_event);
        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start_event, stop_event);
        std::cout<<"GPU Time: "<<gpu_time<<" milliseconds"<<std::endl;

        cudaMemcpy(h_gpu_c, d_C, N*N*sizeof(float), cudaMemcpyDeviceToHost);

        //compare results
        long long temp = 0;
        for(int i = 0; i < N*N; i++)
        {

                temp += (long long)fabs(h_gpu_c[i] - h_C[i]);

        }
        std::cout<<"Difference: "<<temp<<std::endl;
        if(temp == 0)
        {
            std::cout<<"Results are correct"<<std::endl;
        }
        else
        {
            std::cout<<"Results are incorrect"<<std::endl;
        }

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        free(h_A);
        free(h_B);
        free(h_C);
        free(h_gpu_c);

        return 0;

}

Memory allocated on device
CPU Time: 8153.2 milliseconds
GPU Time: 0.318592 milliseconds
Difference: 21428843671
Results are incorrect



In [11]:
%%cuda

/******************************************************************************
 * File: mm_float4_only.cu
 *
 * Demonstrates:
 *   - Tiled matrix multiplication using shared memory
 *   - Vectorized float4 loads (assuming alignment is correct)
 *   - No pinned memory usage (uses normal new[]/delete[] + cudaMalloc/cudaMemcpy)
 *
 * Compile:
 *   nvcc mm_float4_only.cu -o mm_float4_only -std=c++11
 ******************************************************************************/

#include <iostream>
#include <cuda_runtime.h>
#include <cstdlib>
#include <cmath>

// For simplicity, we pick N=512 (a multiple of 16)
static const int N = 512;
// We'll keep tile size "16×16" conceptually, but blockDim.x=4 => each thread loads 4 columns.
static const int TILE_WIDTH = 16;  // tile is 16 wide, 16 tall

//------------------------------------------------------------------------------
// CPU naive mm
//------------------------------------------------------------------------------
void cpu_mm_naive(const float* A, const float* B, float* C, int n)
{
    for (int i = 0; i < n; i++){
        for (int j = 0; j < n; j++){
            float sum = 0.0f;
            for (int k = 0; k < n; k++){
                sum += A[i*n + k] * B[k*n + j];
            }
            C[i*n + j] = sum;
        }
    }
}

//------------------------------------------------------------------------------
// Kernel: Tiled + float4 loads
//   blockDim: (4,16) => 4 threads in x, each loading 4 columns => 16 columns total
//                         16 threads in y => 16 rows
//------------------------------------------------------------------------------
__global__ void mmTiledVec4(const float* __restrict__ A,
                            const float* __restrict__ B,
                            float*       __restrict__ C,
                            int n)
{
    // Shared memory for one tile (16×16) of A and one tile (16×16) of B
    __shared__ float sA[TILE_WIDTH][TILE_WIDTH];
    __shared__ float sB[TILE_WIDTH][TILE_WIDTH];

    // Identify which row & column(s) this thread will handle
    // blockIdx.x, blockIdx.y => which tile
    // threadIdx.x in [0..3], threadIdx.y in [0..15]
    int Row = blockIdx.y * TILE_WIDTH + threadIdx.y;
    // Each thread covers 4 columns:
    //   baseCol = blockIdx.x*16 + threadIdx.x*4
    int baseCol = blockIdx.x * TILE_WIDTH + (threadIdx.x * 4);

    // We'll accumulate partial sums for these 4 columns in a small array
    float sum[4] = {0.0f, 0.0f, 0.0f, 0.0f};

    // how many 16×16 tiles we must iterate over in the K dimension
    int numTiles = n / TILE_WIDTH;  // assume n is multiple of 16

    // For each tile in [0..numTiles-1]
    for (int t = 0; t < numTiles; t++)
    {
        // --------------------------------------------------
        // 1. Load sub-tile of A and B into shared memory
        //    Each thread loads a float4 from A, and a float4 from B
        // --------------------------------------------------

        // A
        //sA[threadIdx.y][threadIdx.x] = A[row * n + (t * TILE_SIZE + threadIdx.x)];
        {   //
            // global col for A => t*TILE_WIDTH + (threadIdx.x*4 .. +3)
            int globalACol = t*TILE_WIDTH + (threadIdx.x * 4);
            const float* srcA = A + (Row*n + globalACol);
            float* dstA = &sA[threadIdx.y][threadIdx.x*4];

            float4 vecA = *reinterpret_cast<const float4*>(srcA);
            *reinterpret_cast<float4*>(dstA) = vecA;
        }

        //sB[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * N + col];
        // B
        {
            // global row for B => t*TILE_WIDTH + threadIdx.y
            int globalBRow = t*TILE_WIDTH + threadIdx.y;
            // columns => baseCol..baseCol+3
            const float* srcB = B + (globalBRow*n + baseCol);
            float* dstB = &sB[threadIdx.y][threadIdx.x*4];

            float4 vecB = *reinterpret_cast<const float4*>(srcB);
            *reinterpret_cast<float4*>(dstB) = vecB;
        }

        __syncthreads();

        // --------------------------------------------------
        // 2. Compute partial sums
        // --------------------------------------------------
        for(int k = 0; k < TILE_WIDTH; k++){
            float aVal = sA[threadIdx.y][k];
            float4 bVal4 = *reinterpret_cast<float4*>(&sB[k][threadIdx.x*4]);
            sum[0] += aVal * bVal4.x;
            sum[1] += aVal * bVal4.y;
            sum[2] += aVal * bVal4.z;
            sum[3] += aVal * bVal4.w;
        }

        __syncthreads();
    }

    // --------------------------------------------------
    // 3. Write out final partial sums
    // --------------------------------------------------
    if(Row < n){
        // We skip boundary checks for columns. If baseCol+3 < n => safe
        float* outC = C + (Row*n + baseCol);
        outC[0] = sum[0];
        outC[1] = sum[1];
        outC[2] = sum[2];
        outC[3] = sum[3];
    }
}

//------------------------------------------------------------------------------
// Main
//------------------------------------------------------------------------------
int main()
{
    // 1. Allocate host memory (regular/paged)
    float* h_A = new float[N*N];
    float* h_B = new float[N*N];
    float* h_C = new float[N*N];
    float* h_Cgpu = new float[N*N];

    // 2. Initialize
    srand(0);
    for(int i=0; i<N*N; i++){
        h_A[i] = float(rand()%10);
        h_B[i] = float(rand()%10);
        h_C[i] = 0.0f;
        h_Cgpu[i] = 0.0f;
    }

    // CPU reference
    cpu_mm_naive(h_A, h_B, h_C, N);

    // 3. Allocate device memory
    float *d_A, *d_B, *d_C;
    size_t nBytes = N*N*sizeof(float);
    cudaMalloc((void**)&d_A, nBytes);
    cudaMalloc((void**)&d_B, nBytes);
    cudaMalloc((void**)&d_C, nBytes);

    // 4. Copy data from Host to Device (synchronous)
    cudaMemcpy(d_A, h_A, nBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, nBytes, cudaMemcpyHostToDevice);

    // 5. Launch the vectorized kernel
    // blockDim => (4,16). => 4 in x => 4 columns per thread, 16 in y => 16 rows
    dim3 block(4,16);
    dim3 grid(N/TILE_WIDTH, N/TILE_WIDTH);
    mmTiledVec4<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    // 6. Copy result back
    cudaMemcpy(h_Cgpu, d_C, nBytes, cudaMemcpyDeviceToHost);

    // 7. Check correctness
    long long diff = 0;
    for(int i=0; i<N*N; i++){
        diff += (long long)fabs(h_Cgpu[i] - h_C[i]);
    }
    if(diff==0) {
        std::cout << "Results match!\n";
    } else {
        std::cout << "Mismatch, diff=" << diff << "\n";
    }

    // Cleanup
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;
    delete[] h_Cgpu;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Results match!

